# 01 — Kiểm tra dữ liệu và rò rỉ

**Câu hỏi:** Bảy bảng nguồn có đủ không, một dòng có đúng là một enrollment không, và feature có nhìn thấy tương lai không?

`studentInfo` là bảng gốc; `studentRegistration` bổ sung ngày đăng ký; `courses` cung cấp độ dài môn. `studentVle` và `studentAssessment` là nhiều dòng trên một enrollment, nên phải tổng hợp **trước khi gộp**. `vle` giải mã loại hoạt động, `assessments` giải mã bài đánh giá.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'clht_code').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from NghienCuu_HCO_VNLS.code_dung_chung.thuc_nghiem import (
    GOC, NGHIEN_CUU, DATA, KET_QUA, HINH, CAC_MOC,
    tao_thu_muc, duong_dan, chia_nhom, nap_du_lieu, chon_dac_trung,
    tien_xu_ly, bo_sung_ty_le, bo_sung_cua_so_vle,
)
tao_thu_muc()
print('Dữ liệu nguồn:', DATA)
print('Kết quả riêng:', KET_QUA)

Dữ liệu nguồn: CLHT_GB_GITHUB\data
Kết quả riêng: CLHT_GB_GITHUB\NghienCuu_HCO_VNLS\ket_qua


In [2]:
import pandas as pd
from clht_code.data import source_manifest
manifest = source_manifest(DATA)
audit = pd.DataFrame([{'source': name, 'columns': len(item['columns']), 'bytes': item['bytes'], 'sha256': item['sha256']} for name, item in manifest.items()])
assert len(audit) == 7
display(audit[['source', 'columns', 'bytes']])
audit.to_csv(KET_QUA / 'kiem_ke_7_bang.csv', index=False, encoding='utf-8-sig')

,source,columns,bytes
0,studentInfo.csv,12,2682912
1,studentRegistration.csv,5,784044
2,assessments.csv,6,5716
3,studentAssessment.csv,5,5690310
4,studentVle.csv,6,453836331
5,vle.csv,6,183746
6,courses.csv,3,526


## 1. Vì sao cutoff không phải là chia 50% số dòng?

Với `courses.module_presentation_length`, ngày cắt là `floor(tỷ lệ × độ dài môn)`. Chỉ giữ `studentVle.date <= cutoff_day`, `studentAssessment.date_submitted <= cutoff_day`; bài đến hạn cũng chỉ tính khi `assessments.date <= cutoff_day`. Bản gộp là một dòng cho `id_student + code_module + code_presentation`.

In [3]:
courses = pd.read_csv(DATA / 'courses.csv')
for cutoff in CAC_MOC:
    courses[f'day_{int(cutoff * 100)}'] = (cutoff * courses.module_presentation_length).floordiv(1).astype(int)
display(courses.head(5))
assert courses['day_50'].between(0, courses.module_presentation_length).all()

,code_module,code_presentation,module_presentation_length,day_50,day_60,day_65,day_70,day_75
0,AAA,2013J,268,134,160,174,187,201
1,AAA,2014J,269,134,161,174,188,201
2,BBB,2013J,268,134,160,174,187,201
3,BBB,2014J,262,131,157,170,183,196
4,BBB,2013B,240,120,144,156,168,180


## 2. Điểm cần kiểm soát

- `final_result` và `risk` **chỉ là nhãn**, không phải feature.
- `date_registration` được dùng vì đã biết ở cutoff; enrollment rút khỏi môn trước cutoff bị loại theo định nghĩa dân số đang theo học. Điều này làm số dòng thay đổi theo cutoff.
- `last_activity_day` phải là ngày lớn nhất **không vượt cutoff**.
- Điểm kiểm tra không được dùng chỉ vì đã có trong CSV: ngày nộp và ngày công bố điểm là hai chuyện khác nhau.
- Thống kê trung bình của toàn bộ dataset trước khi chia tập có thể gây leakage; thí nghiệm này chưa tạo feature chuẩn hóa theo toàn bộ sinh viên cùng môn.

In [4]:
if duong_dan(0.5).exists():
    frame = nap_du_lieu(0.5)
else:
    frame = pd.read_csv(GOC / 'ket_qua' / 'du_lieu_gop_7_bang_cutoff_50.csv')
assert frame[['id_student', 'code_module', 'code_presentation']].duplicated().sum() == 0
assert frame.last_activity_day.dropna().le(frame.loc[frame.last_activity_day.notna(), 'cutoff_day']).all()
assert not {'final_result', 'risk', 'id_student'} & set(chon_dac_trung(frame))
display(pd.DataFrame([{'rows': len(frame), 'features': len(chon_dac_trung(frame)), 'duplicate_enrollment': 0}]))

,rows,features,duplicate_enrollment
0,24599,38,0


**Đọc kết quả:** Assertion đi qua nghĩa là các lỗi rõ ràng về khóa và cutoff đã được chặn; nó **không chứng minh tuyệt đối** dữ liệu không rò rỉ. Việc kiểm tra nguồn của từng feature vẫn là bắt buộc.